# Ijeoma AI — demo on Colab

Launches the Ijeoma AI demo application (Workbench, Chat and Compare pages) on a
Colab GPU and prints a public `gradio.live` URL.

**Prerequisites**
1. Runtime → Change runtime type → **T4 GPU**.
2. A Colab secret named `HF_TOKEN` (key icon in the left sidebar), with notebook
   access enabled, for a Hugging Face account that has accepted the Gemma licence.

Run all cells. The URL is printed in the last cell after three to five minutes
and remains valid for up to 72 hours or until the session ends.


In [ ]:
# Keras selects its backend on first import, so this cell must run first.
import os, sys
if "keras" in sys.modules:
    raise RuntimeError("Keras already imported - Runtime > Restart session, then run from the top.")
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.90"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU - Runtime > Change runtime type > T4"

In [ ]:
!pip install -q -U keras keras-hub "gradio==5.49.1"

In [ ]:
from google.colab import userdata
for name in ("HF_TOKEN", "IjeomaHF", "HUGGINGFACE_TOKEN"):
    try:
        tok = userdata.get(name)
    except Exception:
        tok = None
    if tok:
        os.environ["HF_TOKEN"] = tok
        print("Hugging Face token found in secret", name)
        break
else:
    raise RuntimeError("No Hugging Face token in Colab Secrets. Add HF_TOKEN and enable notebook access.")

In [ ]:
# Fetch the code from GitHub, or from ijeoma_app_bundle.zip on Drive if REPO_URL is empty.
REPO_URL = "https://github.com/morayo0x/ijeoma-ai"
APP_DIR = "/content/ijeoma-ai"

if REPO_URL:
    !rm -rf {APP_DIR} && git clone -q --depth 1 {REPO_URL} {APP_DIR}
else:
    from google.colab import drive
    drive.mount("/content/drive")
    import glob
    hits = glob.glob("/content/drive/MyDrive/Ijeoma/ijeoma_app_bundle.zip") + \
           glob.glob("/content/drive/MyDrive/*/Ijeoma/ijeoma_app_bundle.zip")
    if not hits:
        raise FileNotFoundError("Set REPO_URL, or put ijeoma_app_bundle.zip in the Ijeoma folder on Drive.")
    !rm -rf {APP_DIR} && mkdir -p {APP_DIR} && unzip -q -o "{hits[0]}" -d {APP_DIR}

ADAPTER = f"{APP_DIR}/models/ijeoma_lora_gemma-3-1b-it-r8-lr0.0002-e1.npz"
print("adapter present:", os.path.exists(ADAPTER))

## Launch

The public URL is printed as `Running on public URL: https://….gradio.live`.
Routes: `/` Workbench, `/chat` Chat, `/compare` Compare. The cell runs until
interrupted; stopping it shuts the application down.

In [ ]:
!cd {APP_DIR} && python app/ijeoma_app.py --adapter {ADAPTER} --share --preload-base